In [1]:
import numpy as np
from numba import njit
from scipy.sparse import csr_matrix
from scipy.sparse.linalg import eigs
import multiprocessing as mp
import os
import time
from scipy.optimize import curve_fit

# --- 1. 真值加载 ---
try:
    TRUE_GAMMAS = np.load("riemann_10k_true.npy")
except:
    print("!!! 找不到 riemann_10k_true.npy")
    exit()

# --- 2. 动力学内核 ---
@njit(fastmath=True, nogil=True)
def strict_kernel_v7(u_c, k_static, steps, n_bins):
    x = 0.5
    counts = np.zeros((n_bins, n_bins), dtype=np.float64)
    last_bin = 0
    warmup = 1500000 
    
    for i in range(steps + warmup):
        # 内部微观冷却
        ln_term = np.log(i + 100)
        mu_eff = u_c + k_static / (ln_term**2)
        
        x = 1 - mu_eff * x**2
        
        if x > 1.0: x = 0.999
        elif x < -1.0: x = -0.999
            
        if i > warmup:
            current_bin = int((x + 1.0) / 2.0 * (n_bins - 1))
            if 0 <= current_bin < n_bins:
                counts[last_bin, current_bin] += 1
                last_bin = current_bin
    return counts

# --- 3. 严苛审查 Worker：不准用 Scale！ ---
def gap_sniper_worker(task):
    target_n, k_val = task
    U_C = 1.543689012
    N_BINS = 12000
    STEPS = 2500000 # 250万步确保间距稳定
    
    try:
        t0 = time.time()
        counts = strict_kernel_v7(U_C, k_val, STEPS, N_BINS)
        
        P = csr_matrix(counts)
        row_sums = np.array(P.sum(axis=1)).flatten()
        row_sums[row_sums == 0] = 1.0
        P.data /= row_sums[P.indices]
        
        v0 = np.ones(N_BINS)
        vals, _ = eigs(P, k=150, which='LM', tol=1e-4, v0=v0)
        
        # 正能量态过滤
        pure_vals = vals[(np.abs(vals) > 0.4) & (vals.imag > 1e-7)]
        phases = np.sort(np.angle(pure_vals))
        
        # 🌟 核心修改：绝对间距匹配 (Absolute Gap Matching) 🌟
        if len(phases) >= 80:
            check_len = min(80, len(phases)) # 取前80个点测宽度
            sim_width = phases[check_len-1] - phases[0]
            true_width = TRUE_GAMMAS[target_n + check_len - 1] - TRUE_GAMMAS[target_n]
            
            # 绝对误差：不仅要形状像，宽度必须一模一样！
            gap_err = np.abs(sim_width - true_width)
        else:
            gap_err = 999.9
            
        return target_n, k_val, gap_err, time.time()-t0
    except Exception as e:
        return target_n, k_val, 999.9, 0.0

# --- 4. 战役调度与自动拟合 ---
if __name__ == "__main__":
    print(f"="*60)
    print(f"🚀 V7 绝对零容忍测绘：寻找真实的 Beta")
    print(f"="*60)
    
    # 32 个能量站 (100 -> 10000)
    n_stations = np.linspace(100, 9900, 32).astype(int)
    # 32 个探测器 (高密度覆盖 5.0 到 9.0)
    k_probes = np.linspace(5.0, 9.0, 32)
    
    tasks = [(n, k) for n in n_stations for k in k_probes]
    print(f">>> 总任务数: {len(tasks)} (256核集群饱和打击准备完毕)")
    
    start_time = time.time()
    results = []
    
    # 用 200 并发，留点内存余量给系统
    with mp.Pool(processes=200) as pool:
        for i, res in enumerate(pool.imap_unordered(gap_sniper_worker, tasks)):
            results.append(res)
            if (i+1) % 100 == 0:
                print(f"   已完成 {i+1}/{len(tasks)} 探针测试...")
                
    print(f"\n✅ 饱和打击完成！耗时: {(time.time()-start_time)/60:.2f} 分钟")
    
    # --- 5. 自动反演真理参数 ---
    print("\n>>> 正在提取无缩放条件下的最优物理路径...")
    best_ks = {}
    for res in results:
        target_n, k_val, gap_err, _ = res
        if target_n not in best_ks or gap_err < best_ks[target_n][1]:
            best_ks[target_n] = (k_val, gap_err)
            
    fit_n = np.array(sorted(best_ks.keys()))
    fit_k = np.array([best_ks[n][0] for n in fit_n])
    
    # 拟合您的对数物理公式：k(n) = k0 + beta / ln(n)
    def true_physics_model(n, k0, beta):
        return k0 + beta / np.log(n)
        
    popt, _ = curve_fit(true_physics_model, fit_n, fit_k, p0=[4.7, 10.0])
    
    print("\n" + "🔥"*20)
    print(f"👑 真理参数脱水版 (无 Scale 作弊):")
    print(f"   基态响应 (k0)   = {popt[0]:.6f}")
    print(f"   冷却惯性 (beta) = {popt[1]:.6f}   <-- 看看是不是接近 10.13！")
    print("🔥"*20)

🚀 V7 绝对零容忍测绘：寻找真实的 Beta
>>> 总任务数: 1024 (256核集群饱和打击准备完毕)
   已完成 100/1024 探针测试...
   已完成 200/1024 探针测试...
   已完成 300/1024 探针测试...
   已完成 400/1024 探针测试...
   已完成 500/1024 探针测试...
   已完成 600/1024 探针测试...
   已完成 700/1024 探针测试...
   已完成 800/1024 探针测试...
   已完成 900/1024 探针测试...
   已完成 1000/1024 探针测试...

✅ 饱和打击完成！耗时: 39.80 分钟

>>> 正在提取无缩放条件下的最优物理路径...

🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥
👑 真理参数脱水版 (无 Scale 作弊):
   基态响应 (k0)   = 2.980559
   冷却惯性 (beta) = 23.890557   <-- 看看是不是接近 10.13！
🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥
